# Fine-tuning BERT: venda ou suporte

Este notebook treina o **BERTimbau**, um BERT pré-treinado para português, para classificar mensagens de clientes como `venda` ou `suporte`.

> No Google Colab, selecione **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU** antes de começar.

## 1. Instalação das dependências

In [ ]:
%pip install -q "torch>=2.2,<3" "transformers>=4.48,<5" "datasets>=3.2,<5" "accelerate>=1.2,<2" "scikit-learn>=1.4,<2"

## 2. Imports e configuração

O mapeamento das classes fica salvo na configuração do modelo, evitando ambiguidades durante a inferência.

In [ ]:
import json
import random
from pathlib import Path

import numpy as np
import torch
from datasets import Dataset
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    ConfusionMatrixDisplay,
    precision_recall_fscore_support,
)
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

SEED = 42
MODEL_NAME = "neuralmind/bert-base-portuguese-cased"
OUTPUT_DIR = "modelo_atendimento"
LABEL2ID = {"suporte": 0, "venda": 1}
ID2LABEL = {value: key for key, value in LABEL2ID.items()}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo: {device}")

## 3. Upload e validação dos dados

No Colab, a célula abre o seletor de arquivos se os JSONL ainda não estiverem disponíveis. Fora do Colab, coloque os arquivos ao lado do notebook.

In [ ]:
required_files = [Path("treino.jsonl"), Path("teste.jsonl")]

if not all(path.is_file() for path in required_files):
    try:
        from google.colab import files
        print("Selecione treino.jsonl e teste.jsonl")
        files.upload()
    except ImportError:
        missing = [str(path) for path in required_files if not path.is_file()]
        raise FileNotFoundError(
            "Copie os arquivos para a pasta do notebook: " + ", ".join(missing)
        )

def load_jsonl(path):
    records = []
    with Path(path).open(encoding="utf-8") as file:
        for line_number, raw_line in enumerate(file, start=1):
            if not raw_line.strip():
                continue
            try:
                item = json.loads(raw_line)
            except json.JSONDecodeError as error:
                raise ValueError(f"{path}, linha {line_number}: JSON inválido") from error

            prompt = item.get("prompt")
            label = str(item.get("completion", "")).strip().lower()
            if not isinstance(prompt, str) or not prompt.strip():
                raise ValueError(f"{path}, linha {line_number}: prompt vazio ou ausente")
            if label not in LABEL2ID:
                raise ValueError(
                    f"{path}, linha {line_number}: classe '{label}' inválida"
                )
            records.append({"text": prompt.strip(), "labels": LABEL2ID[label]})

    if not records:
        raise ValueError(f"{path}: nenhum exemplo encontrado")
    if len({row["labels"] for row in records}) != 2:
        raise ValueError(f"{path}: o arquivo precisa conter as duas classes")
    return records

train_dataset = Dataset.from_list(load_jsonl("treino.jsonl"))
test_dataset = Dataset.from_list(load_jsonl("teste.jsonl"))

def class_counts(dataset):
    values, counts = np.unique(dataset["labels"], return_counts=True)
    return {ID2LABEL[int(value)]: int(count) for value, count in zip(values, counts)}

print(f"Treino: {len(train_dataset)} exemplos — {class_counts(train_dataset)}")
print(f"Teste:  {len(test_dataset)} exemplos — {class_counts(test_dataset)}")

## 4. Tokenização

O padding é dinâmico: cada lote é preenchido somente até o tamanho do maior texto daquele lote, economizando memória.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=128)

tokenized_train = train_dataset.map(
    tokenize, batched=True, remove_columns=["text"]
)
tokenized_test = test_dataset.map(
    tokenize, batched=True, remove_columns=["text"]
)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## 5. Modelo, métricas e treinamento

O melhor checkpoint é selecionado pelo F1 ponderado. Isso considera o desempenho das duas classes e é mais robusto que observar apenas a acurácia quando há desbalanceamento.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    label2id=LABEL2ID,
    id2label=ID2LABEL,
)

def compute_metrics(eval_prediction):
    logits, labels = eval_prediction
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="weighted", zero_division=0
    )
    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=SEED,
    data_seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

## 6. Avaliação detalhada

In [ ]:
metrics = trainer.evaluate()
for name in ["eval_loss", "eval_accuracy", "eval_precision", "eval_recall", "eval_f1"]:
    if name in metrics:
        print(f"{name}: {metrics[name]:.4f}")

prediction_output = trainer.predict(tokenized_test)
predictions = np.argmax(prediction_output.predictions, axis=-1)
references = np.asarray(test_dataset["labels"])

print("\nRelatório por classe:")
print(classification_report(
    references,
    predictions,
    labels=[0, 1],
    target_names=[ID2LABEL[0], ID2LABEL[1]],
    zero_division=0,
))

ConfusionMatrixDisplay.from_predictions(
    references,
    predictions,
    display_labels=[ID2LABEL[0], ID2LABEL[1]],
    cmap="Blues",
);

## 7. Salvamento e inferência

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

metricas_serializaveis = {
    key: float(value)
    for key, value in metrics.items()
    if isinstance(value, (int, float, np.integer, np.floating))
}
with Path(OUTPUT_DIR, "metricas.json").open("w", encoding="utf-8") as file:
    json.dump(metricas_serializaveis, file, ensure_ascii=False, indent=2)

print(f"Modelo salvo em {OUTPUT_DIR}")

In [ ]:
def classify(message):
    model.eval()
    inputs = tokenizer(
        message, return_tensors="pt", truncation=True, max_length=128
    ).to(model.device)
    with torch.inference_mode():
        probabilities = torch.softmax(model(**inputs).logits, dim=-1)[0]
    predicted_id = int(torch.argmax(probabilities).item())
    return {
        "mensagem": message,
        "classe": model.config.id2label[predicted_id],
        "confianca": round(float(probabilities[predicted_id]), 4),
    }

classify("Gostaria de comprar uma televisão nova")